In [1]:
import cv2
import sys
import glob
import numpy as np
import pandas as pd 
import matplotlib
import matplotlib.pyplot as plt
sys.path.append(r"/Users/johnmarshall/Documents/Analysis/post_cnmfe_analysis")
import dlc_utils as dlc
import os
import datetime
import math
import time 
from tqdm import tqdm
from matplotlib.dates import DateFormatter, date2num
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:98% !important; }</style>"))

/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_55486/3730779655.py:16: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
#save path
base_dir = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/SCI_analysis/velocityAlignmentAnalysis/IntroFigure/DIO_r2.26_12_58_49/'
savePath = base_dir 

#params 
behavCamFrameRate=15
miniscopeCamFrameRate=20
downSampleFactorForVelocity = '.2S'
medianFilter = '1S'
sessionPath = savePath+'DIO_r2.26_12_58_49_combined_LocationOutput.csv'

print(sessionPath.split(os.sep)[:-1])
# load eZ track output and behavior camera timestamps from miniscope software 
ezTrackOutput = pd.read_csv(sessionPath)
timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStamps.csv', delimiter=',')
miniscope_timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStampsMiniscope.csv', delimiter=',')
    
timestampfile_td = timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(timestampfile)-1)*(1/behavCamFrameRate), len(timestampfile)), unit='s'), drop=False)
  
miniscopetimestamp_td_originalHz = miniscope_timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(miniscope_timestampfile)-1)*(1/miniscopeCamFrameRate), len(miniscope_timestampfile)), unit='s'), drop=False)

['', 'Users', 'johnmarshall', 'Documents', 'Analysis', 'miniscope_analysis', 'SCI_analysis', 'velocityAlignmentAnalysis', 'IntroFigure', 'DIO_r2.26_12_58_49']


In [3]:
timestampfile_td.head()

,Frame Number,Time Stamp (ms),Buffer Index
0 days 00:00:00,0,-12,0
0 days 00:00:00.066666667,1,52,0
0 days 00:00:00.133333333,2,116,0
0 days 00:00:00.200000,3,181,0
0 days 00:00:00.266666667,4,260,0


In [4]:
miniscopetimestamp_td_originalHz.head()

,Frame Number,Time Stamp (ms),Buffer Index
0 days 00:00:00,0,-27,0
0 days 00:00:00.050000,1,9,0
0 days 00:00:00.100000,2,40,0
0 days 00:00:00.150000,3,73,0
0 days 00:00:00.200000,4,106,0


In [5]:
behavCam_frames = []
sys_clock_behavCam = []
#create "key" for aligning miniscope frames to timestamp file
#then create behavior TD and align
for msCam_frame in tqdm(range(0, len(miniscopetimestamp_td_originalHz['Frame Number']))):
    #get sys clock time of each miniscope recorded frame
    #sys_clock_msCam = time_stamps['sysClock'].loc[msCam_frame]
    #find behav cam frame closest to sys clock time of ms frame
    behavCam_frame = list(timestampfile_td.iloc[(timestampfile_td['Time Stamp (ms)']-miniscopetimestamp_td_originalHz['Time Stamp (ms)'].iloc[msCam_frame]).abs().argsort()[:1]].index)[0]
    #this is the behavCamIndex that is closest to the corresponding miniscope frame 
    behavCam_frames.append(behavCam_frame)
    sys_clock_behavCam.append(timestampfile_td.loc[behavCam_frame]['Time Stamp (ms)'])

behavCamIdxToAlign = [timestampfile_td.index.get_loc(idx) for idx in behavCam_frames]
#ezTrackOutput
miniscopetimestamp_td_originalHz['closestBehavCamFrameIdx'] = behavCamIdxToAlign

X_coor=[]
Y_coor=[]
Distance_px=[] 
Distance_cm=[]

for i in tqdm(miniscopetimestamp_td_originalHz['closestBehavCamFrameIdx'].values):
    X_coor.append(ezTrackOutput.loc[i]['X'])
    Y_coor.append(ezTrackOutput.loc[i]['Y'])
    Distance_px.append(ezTrackOutput.loc[i]['Distance_px'])
    Distance_cm.append(ezTrackOutput.loc[i]['Distance_cm'])
    
miniscopetimestamp_td_originalHz['X_coor'] = X_coor
miniscopetimestamp_td_originalHz['Y_coor'] = Y_coor
miniscopetimestamp_td_originalHz['Distance_px'] = Distance_px
miniscopetimestamp_td_originalHz['Distance_cm'] = Distance_cm

100%|██████████████████████████████████| 39446/39446 [00:02<00:00, 14827.50it/s]


In [6]:
# calculate cmPerPixel
cmPerPixel = ezTrackOutput['Distance_cm'][1]/ezTrackOutput['Distance_px'][1]

print(cmPerPixel)
def euclideanDistance(row):
    return(math.sqrt(row['X_coor']**2+row['Y_coor']**2))  
def euclideanDistancePixels(row, cmPerPixel):
    return(math.sqrt(row['X_coor']**2+row['Y_coor']**2)*cmPerPixel)
def calcVelocity(row, cmPerPixel, samplingRate):
    #euclidean distance divided by sampling rate, multiplied by cm/pixel
    return((math.sqrt(row['X_coor']**2+row['Y_coor']**2)/samplingRate)*cmPerPixel)
    
# 
samplingRateOrig = timestampfile['Time Stamp (ms)'].diff().mean()/1000
print('orig sampling rate')
print(samplingRateOrig)
miniscopetimestamp_td_originalHz['Velocity'] = miniscopetimestamp_td_originalHz.diff().apply(lambda row : calcVelocity(row, cmPerPixel, samplingRateOrig), axis=1)
miniscopetimestamp_td_originalHz['Distance'] = miniscopetimestamp_td_originalHz.diff().apply(lambda row : euclideanDistancePixels(row, cmPerPixel), axis=1)

## in doing the downsampling, break apart X_coor, Y_coor columns to to .mean() instead of .bfill()

# downsample to .1S
miniscopeTimeStampsDownsampled_1 = miniscopetimestamp_td_originalHz.resample('.1S').bfill()
X_Y_Downsampled_1 = miniscopetimestamp_td_originalHz[['X_coor', 'Y_coor']].resample('.1S').mean()
miniscopeTimeStampsDownsampled_1[['X_coor', 'Y_coor']] = X_Y_Downsampled_1

# downsample to .2S
miniscopeTimeStampsDownsampled_2 = miniscopetimestamp_td_originalHz.resample('.2S').bfill()
X_Y_Downsampled_2 = miniscopetimestamp_td_originalHz[['X_coor', 'Y_coor']].resample('.2S').mean()
miniscopeTimeStampsDownsampled_2[['X_coor', 'Y_coor']] = X_Y_Downsampled_2

#
samplingRate_1 = 0.1
miniscopeTimeStampsDownsampled_1['Velocity'] = miniscopeTimeStampsDownsampled_1.diff().apply(lambda row : calcVelocity(row, cmPerPixel, samplingRate_1), axis=1)
miniscopeTimeStampsDownsampled_1['Distance'] = miniscopeTimeStampsDownsampled_1.diff().apply(lambda row : euclideanDistancePixels(row, cmPerPixel), axis=1)

#
samplingRate_2 = 0.2
miniscopeTimeStampsDownsampled_2['Velocity'] = miniscopeTimeStampsDownsampled_2.diff().apply(lambda row : calcVelocity(row, cmPerPixel, samplingRate_2), axis=1)
miniscopeTimeStampsDownsampled_2['Distance'] = miniscopeTimeStampsDownsampled_2.diff().apply(lambda row : euclideanDistancePixels(row, cmPerPixel), axis=1)

# save these for comparison
miniscopetimestamp_td_originalHz.to_csv(savePath+os.path.split(sessionPath)[0].split('/')[-1]+'velocity_origHz.csv')
miniscopeTimeStampsDownsampled_1.to_csv(savePath+os.path.split(sessionPath)[0].split('/')[-1]+'velocity_resampled_1.csv')

miniscopeTimeStampsDownsampled_2.to_csv(savePath+os.path.split(sessionPath)[0].split('/')[-1]+'velocity_resampled_2.csv')

0.10946206071673621
orig sampling rate
0.06666708618606525


/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_55486/3233831459.py:23: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  miniscopeTimeStampsDownsampled_1 = miniscopetimestamp_td_originalHz.resample('.1S').bfill()
/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_55486/3233831459.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  X_Y_Downsampled_1 = miniscopetimestamp_td_originalHz[['X_coor', 'Y_coor']].resample('.1S').mean()
/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_55486/3233831459.py:28: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  miniscopeTimeStampsDownsampled_2 = miniscopetimestamp_td_originalHz.resample('.2S').bfill()
/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_55486/3233831459.py:29: FutureWarning: 'S' is deprecated and will be removed in a future version, please use '

In [19]:
#ezTrackOutput

In [17]:
miniscopetimestamp_td_originalHz

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px,Distance_cm,Velocity,Distance
0 days 00:00:00,0,-27,0,0,68.155816,328.103448,0.000000,0.000000,NaN,NaN
0 days 00:00:00.050000,1,9,0,0,68.155816,328.103448,0.000000,0.000000,0.000000,0.000000
0 days 00:00:00.100000,2,40,0,1,64.492886,326.236045,4.111478,0.450051,6.750721,0.450051
0 days 00:00:00.150000,3,73,0,1,64.492886,326.236045,4.111478,0.450051,0.000000,0.000000
0 days 00:00:00.200000,4,106,0,2,66.804493,327.652632,2.711133,0.296766,4.451466,0.296766
...,...,...,...,...,...,...,...,...,...,...
0 days 00:32:52.050000,39441,1324149,0,19862,198.037454,286.798201,5.576928,0.610462,0.000000,0.000000
0 days 00:32:52.100000,39442,1324182,0,19863,210.049444,276.462917,15.846325,1.734571,26.018407,1.734571
0 days 00:32:52.150000,39443,1324217,0,19863,210.049444,276.462917,15.846325,1.734571,0.000000,0.000000
0 days 00:32:52.200000,39444,1324250,0,19864,204.301510,279.413439,6.460985,0.707233,10.608424,0.707233


In [16]:
miniscopeTimeStampsDownsampled_2[0:20]

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px,Distance_cm,Velocity,Distance
0 days 00:00:00,0,-27,0,0,66.324351,327.169747,0.000000,0.000000,NaN,NaN
0 days 00:00:00.200000,4,106,0,2,67.600826,327.827212,2.711133,0.296766,0.785852,0.157170
0 days 00:00:00.400000,8,241,0,4,67.435817,327.399474,0.740024,0.081005,0.250921,0.050184
0 days 00:00:00.600000,12,374,0,6,67.209296,327.933604,2.211678,0.242095,0.317538,0.063508
0 days 00:00:00.800000,16,508,0,8,68.667300,328.103403,3.282959,0.359359,0.803374,0.160675
0 days 00:00:01,20,644,0,10,68.156670,327.421919,1.096748,0.120052,0.466070,0.093214
0 days 00:00:01.200000,24,777,0,12,67.758260,327.396039,1.622686,0.177623,0.218514,0.043703
0 days 00:00:01.400000,28,912,0,14,69.565070,327.669429,6.637563,0.726561,1.000142,0.200028
0 days 00:00:01.600000,32,1047,0,16,68.041961,327.055255,4.925833,0.539192,0.898835,0.179767
0 days 00:00:01.800000,36,1181,0,18,66.770530,327.168402,0.253215,0.027717,0.698617,0.139723


In [8]:
## check alignment with video overlay 
alignedTracesLocationData = miniscopetimestamp_td_originalHz[['closestBehavCamFrameIdx', 'X_coor', 'Y_coor', 'Velocity']]
locationReIndexedToBehaviorData = (
    alignedTracesLocationData
      .groupby('closestBehavCamFrameIdx')           # group by frame-idx
      .mean(numeric_only=True)                     # mean of only number-dtype cols
)

In [9]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────

dirPath      = savePath
rotatedVideo = 'video_output.avi'
input_vid    = os.path.join(dirPath, rotatedVideo)
output_vid   = os.path.join(dirPath, 'with_velocity_overlay.mp4')

vel_array = locationReIndexedToBehaviorData['Velocity'].to_numpy()
x_pos     = locationReIndexedToBehaviorData['X_coor'].to_numpy()
y_pos     = locationReIndexedToBehaviorData['Y_coor'].to_numpy()

bar_h    = 50  # height of black bar

# ── SET UP I/O ─────────────────────────────────────────────────────────────────

cap      = cv2.VideoCapture(input_vid)
fps      = cap.get(cv2.CAP_PROP_FPS)
w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out    = cv2.VideoWriter(output_vid, fourcc, fps, (w, h + bar_h))

# text & dot styling
font        = cv2.FONT_HERSHEY_SIMPLEX
font_scale  = 1
thickness   = 2
margin      = 10
dot_radius  = 5

# ── PROCESS FRAMES ──────────────────────────────────────────────────────────────

for frame_idx in tqdm(range(n_frames), desc="Adding overlay"):
    ret, frame = cap.read()
    if not ret:
        break

    # 1) overlay the dot on the ORIGINAL frame at (x_pos, y_pos)
    #    clamp coordinates to [0, w-1] and [0, h-1]
    xm = int(np.clip(x_pos[frame_idx], 0, w - 1))
    ym = int(np.clip(y_pos[frame_idx], 0, h - 1))
    cv2.circle(frame, (xm, ym), dot_radius, (255, 255, 255), -1)

    # 2) create the black bar underneath, then stack
    bar       = np.zeros((bar_h, w, 3), dtype=np.uint8)
    new_frame = np.vstack((frame, bar))

    # 3) overlay the velocity text in the bar (left‐aligned)
    vel_text = f"{vel_array[frame_idx]:.2f} cm/s"
    (tw, th_text), _ = cv2.getTextSize(vel_text, font, font_scale, thickness)
    text_x = margin
    text_y = h + margin + th_text
    cv2.putText(
        new_frame,
        vel_text,
        (text_x, text_y),
        font,
        font_scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA
    )

    out.write(new_frame)

# ── CLEAN UP ────────────────────────────────────────────────────────────────────

cap.release()
out.release()
print("Done – saved with overlay to:", output_vid)

OpenCV: FFMPEG: tag 0x44495658/'XVID' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
Adding overlay: 100%|███████████████████| 19865/19865 [00:16<00:00, 1228.00it/s]

Done – saved with overlay to: /Users/johnmarshall/Documents/Analysis/miniscope_analysis/SCI_analysis/velocityAlignmentAnalysis/IntroFigure/DIO_r2.26_12_58_49/with_velocity_overlay.mp4
